# Confirmation Bias Project
## Preparing data for group level analyses
#### __Experiment 3:__ 

2 sequences with same mean DV. Control analyses to compare whether differences in drift across repetitions depend on participants confidence/metacognitive abilities. We also manipulate that in some blocks the gratings are exactly the same and in others the gratings are different (but with similar mean DVs across repetitions)

In [1]:
exp_name = 'CJ'

##### Import important functions and libraries

In [2]:
%matplotlib inline

In [3]:
#%matplotlib qt
%matplotlib inline

In [4]:
import os, glob
import numpy as np
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import scipy
import scipy.stats as stats
from scipy import signal
import seaborn as sns
from statsmodels.formula.api import ols
import statsmodels.formula.api as smf
from statsmodels.stats.anova import AnovaRM
import pingouin as pg
from statsmodels.stats.multicomp import (pairwise_tukeyhsd, MultiComparison)
from matplotlib.lines import Line2D
import statsmodels as sms
#import ptitprince as pt
pd.options.display.max_columns = None # display all the columns in pandas dataframe


In [5]:
#! pip install numpyro
import numpy as np
import arviz as az
#import numpyro
from metadPy.bayesian import hmetad
from metadPy.mle import metad, fit_metad
#from metadPy import load_dataset
from metadPy.sdt import criterion, dprime, rates, roc_auc, scores

# Set the number of cores used by Numpyro here
#numpyro.set_host_device_count(2)
from metadPy.utils import responseSimulation
simulation = responseSimulation(d=1, metad=1.5, c=0, 
                                nRatings=4, nTrials=200)

ModuleNotFoundError: No module named 'metadPy'

##### Important functions

In [6]:
def log_reg_fit(x, f): # this function was used to calculate the log linear regression between two vectors
    reg = smf.glm(formula = f, data = x, family=sm.families.Binomial()).fit()
    params = reg.params
    PSE = - reg.params.Intercept/reg.params[1]
    # concatenating parameters
    out = pd.DataFrame({'intercept':[params[0]] , 'weight':[params[1]],'PSE': PSE})
    return out #intercept + weight

def cartesian(arrays, out=None):
    """
    Examples
    --------
    >>> cartesian(([1, 2, 3], [4, 5], [6, 7]))
    """

    arrays = [np.asarray(x) for x in arrays]
    dtype = arrays[0].dtype

    n = np.prod([x.size for x in arrays])
    if out is None:
        out = np.zeros([n, len(arrays)], dtype=dtype)

    #m = n / arrays[0].size
    m = int(n / arrays[0].size) 
    out[:,0] = np.repeat(arrays[0], m)
    if arrays[1:]:
        cartesian(arrays[1:], out=out[0:m, 1:])
        for j in range(1, arrays[0].size):
        #for j in xrange(1, arrays[0].size):
            out[j*m:(j+1)*m, 1:] = out[0:m, 1:]
    return out

# lets bin the confidence continuous variable into 10 bins
def bin_continuous_variable(series, num_bins=6, binning_method='equal_width', labels=None):
    """
    Bins a continuous variable in a pandas Series into a specified number of bins.

    Args:
        series (pd.Series): The continuous variable to bin.
        num_bins (int): The number of bins to create (default is 6).
        binning_method (str): The binning method to use. Options are 'equal_width' (default) and 'quantile'.
        labels (list): Optional labels for the bins. If None, default labels are used.

    Returns:
        pd.Series: A pandas Series containing the bin labels.
    """

    if binning_method == 'equal_width':
        # Equal-width binning
        bins = np.linspace(series.min(), series.max(), num_bins + 1)
        binned_series = pd.cut(series, bins, labels=labels, include_lowest=True)
    elif binning_method == 'quantile':
        # Quantile binning
        binned_series = pd.qcut(series, num_bins, labels=labels)
    else:
        raise ValueError("Invalid binning_method. Choose 'equal_width' or 'quantile'.")

    return binned_series



### Data & variables

In [8]:
import pickle

# Creamos una clase "falsa" (Dummy) para engañar a pickle
class DummyObject:
    def __init__(self, *args, **kwargs):
        pass
    
    # Esto permite que pickle guarde los datos originales dentro de este objeto falso
    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)
        elif isinstance(state, tuple) and len(state) == 2 and isinstance(state[1], dict):
            self.__dict__.update(state[1])

# Creamos un lector personalizado que intercepta las llamadas a módulos que no tienes
class SafePsychoPyUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Si pickle busca módulos problemáticos, le damos nuestra clase falsa
        if 'psychopy' in module or 'json_tricks' in module:
            return DummyObject
        
        # Si son objetos normales de Python (listas, diccionarios, pandas), los carga normal
        return super().find_class(module, name)

In [ ]:

current_dir = os.getcwd()
# Go up 3 levels ("../../../") and then into your target folder
results_path = os.path.abspath(os.path.join(current_dir, "../../../Behav_data/Behavioral/condcisionCJ"))

os.chdir(results_path) # change the current working directory to the results path 

all_df = pd.DataFrame([]) # concatenate all behav subject data together
ddata = pd.DataFrame([]) # concatenate here behav data + dv + orientations

# Initialize a list to hold the demographic info for each subject
subj_info_list = []

nsubj = 0 # initialize subjects counter

for file in sorted(glob.glob("*.psydat")): 
    subjdata = pd.DataFrame([]) # initialize individual subject data variable container
    pfile =  open(os.path.join(results_path, file),"rb")
    nsubj = nsubj + 1
    
    dat = SafePsychoPyUnpickler(pfile).load()
    
    for block in dat['main_exp']['Exp_blocks']:
        dvdata = pd.DataFrame(signal.sawtooth(4 * (block['trial_orientations']), 0.5),columns=['d1','d2','d3','d4','d5','d6'])
        ddata = pd.concat([block['data'], dvdata, block['trial_orientations']], axis = 1)
        subjdata = pd.concat([subjdata, ddata], axis = 0)
    
    subjdata.insert(0, 'npar', nsubj)
    all_df = pd.concat([all_df, subjdata], axis = 0) #concatenate each new subject
    
    # Safely extract subject info and append to our list
    # We use .get() in case a key is missing for some participants
    subj_info = {
        'npar': nsubj,
        'gender': dat['subjInfo'].get('gender (M/F)', None),
        'age': dat['subjInfo'].get('age', None)
    }
    subj_info_list.append(subj_info)

# Create the new dataframe from the list after the loop finishes
subj_info_df = pd.DataFrame(subj_info_list)

# You can view the new dataframe like this:
display(subj_info_df.head())

all_df.head(10)


/var/folders/rx/vfyhhpqd7q172_xk2p75xp7w0000gn/T/ipykernel_7427/413132327.py:23: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dat = SafePsychoPyUnpickler(pfile).load()
/var/folders/rx/vfyhhpqd7q172_xk2p75xp7w0000gn/T/ipykernel_7427/413132327.py:23: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dat = SafePsychoPyUnpickler(pfile).load()
/var/folders/rx/vfyhhpqd7q172_xk2p75xp7w0000gn/T/ipykernel_7427/413132327.py:23: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dat = SafePsychoPyUnpickler(pfile).load()
/var/folders/rx/vfyhhpqd7q172_xk2p75xp7w0000gn/T/ipykernel_7427/413132327

,npar,gender,age
0,1,F,19
1,2,F,19
2,3,F,19
3,4,F,18
4,5,F,18


,npar,subj,nblock,ntrial,nrep,blocktype,trial_type,cond,DV,resp,r_map,correct,confi,RT,d1,d2,d3,d4,d5,d6,o1,o2,o3,o4,o5,o6
0,1,s01,0,0,0,nonrepeat,False,1,0.08,1.0,45,1,0.10,2.044453,0.828372,-0.956710,0.360338,-0.087842,0.939380,-0.549273,0.718,0.017,2.105,1.929,2.380,0.177
1,1,s01,0,0,1,nonrepeat,False,1,0.08,1.0,0,1,0.35,2.114065,-0.186164,-0.022671,0.287481,0.530953,0.415324,-0.616519,2.822,1.187,2.636,2.172,1.015,2.991
2,1,s01,0,1,0,nonrepeat,False,-1,-0.01,1.0,45,-1,0.75,1.752747,0.840586,-0.430107,0.129599,0.858411,-0.717341,-0.844146,0.848,1.347,2.698,0.841,0.111,1.632
3,1,s01,0,1,1,nonrepeat,False,-1,-0.01,-1.0,45,1,-0.15,1.981836,-0.668958,0.911887,0.596642,0.846197,-0.812598,-0.861971,0.130,0.820,0.627,0.725,3.068,1.625
4,1,s01,0,2,0,nonrepeat,False,-1,-0.11,-1.0,45,1,0.40,2.481966,-0.637881,0.462198,-0.473916,0.087865,-0.446895,0.491718,1.713,2.145,2.935,1.998,1.788,0.985
5,1,s01,0,2,1,nonrepeat,False,-1,-0.11,1.0,45,-1,0.70,1.398619,-0.197340,0.444372,0.665916,-0.903752,0.329781,-0.864518,1.886,2.138,2.225,1.533,2.093,1.624
6,1,s01,0,3,0,nonrepeat,False,1,0.11,1.0,45,1,0.60,1.405562,0.519729,-0.601240,-0.049645,0.634840,0.319076,-0.260012,0.974,2.985,1.944,0.642,0.518,2.851
7,1,s01,0,3,1,nonrepeat,False,1,0.11,1.0,45,1,0.25,1.301392,0.518220,-0.770817,0.939380,0.513127,0.375617,-0.952654,2.167,0.090,2.380,2.165,2.111,3.123
8,1,s01,0,4,0,nonrepeat,False,-1,-0.01,-1.0,0,1,-0.10,3.336132,0.417352,-0.934829,-0.528901,-0.118918,0.922592,0.326197,2.585,3.116,0.185,0.346,0.755,1.050
9,1,s01,0,4,1,nonrepeat,False,-1,-0.01,1.0,45,-1,0.70,1.197235,0.397498,-0.783031,-0.578322,0.823279,-0.488158,0.678648,1.022,1.656,2.976,0.716,0.201,2.230


In [11]:
df = all_df.copy() # copy tge variable

# relabeling some variables
df['cond'] = all_df['cond'].replace([-1], 0)
df['correct'] = all_df['correct'].replace([-1], 0)
df.insert(5, 'cond-1', 0)    # 0 = previous Diag / 1 = previous Card
df['cond-1'] = df['cond'].shift(1, fill_value  = 0)

# Inserting deci variable
df.insert(8, 'deci', 0)
crit1 = (df['cond']  > 0) & (df['correct'] == 1);    # 0 = diagonal       
crit2 = (df['cond']  < 0) & (df['correct'] == 0);            
crit3 = (df['cond'] == 0) & (df['correct'] == 0);
df.loc[crit1 | crit2 | crit3, 'deci'] = 1 

# Inserting the new necessary columns coding trials properly
df.insert(7, 'deci-3', 0)    # 0 = 2 previous Diag / 1 = previous Card
df.insert(8, 'deci-2', 0)    # 0 = previous Diag / 1 = previous Card
df.insert(9, 'deci-1', 0)    # 0 = previous Diag / 1 = previous Card
df.insert(14, 'corr-1', 0) # O = incorrect / 1 = correct
df.insert(7, 'rDV', 0) # rDV (real Decision Variable)

# Recoding variables
df['deci-1'] = df['deci'].shift(1, fill_value  = 0) # deci in trial n-1
df['deci-2'] = df['deci-1'].shift(1, fill_value  = 0) # deci in trial n-2
df['deci-3'] = df['deci-2'].shift(1, fill_value  = 0) # deci in trial n-2
df['corr-1'] = df['correct'].shift(1, fill_value  = 0) # correct in trial n-1
df.drop(columns=['trial_type'], inplace = True)
df.rename(columns = {'blocktype' : 'trial_type'}, inplace = True)
df['rDV'] = np.mean(df.iloc[:,20:26], axis = 1) # average DV
df.head(50)

# Recode the confidence variable
df['confi'] = bin_continuous_variable(df.confi, num_bins=10, labels=[0,1,2,3,4,5,6,7,8,9])

df.reset_index(drop=True, inplace=True)

## df['nrep'] = df['nrep'].astype('category')
df['npar'] = df['npar'].astype('category')
df['nblock'] = df['nblock'].astype('category')
df['cond-1'] = df['cond-1'].astype('category')
df['deci-1'] = df['deci-1'].astype('category')
df['deci-2'] = df['deci-2'].astype('category')
df['deci-3'] = df['deci-3'].astype('category')
df.insert(0, 'exp_ID', exp_name) # rDV (real Decision Variable)

In [12]:
df.npar.unique() # check the number of subjects

[1, 2, 3, 4, 5, ..., 34, 35, 36, 37, 38]
Length: 38
Categories (38, int64): [1, 2, 3, 4, ..., 35, 36, 37, 38]

In [16]:
%%capture --no-display

nprows = 8 
npcols = 6

nrep_labels = np.unique(df.nrep) #nrep
npar_labels = np.unique(df.npar) #npar7

fig, axes = plt.subplots(nprows, npcols, figsize=(18, 24))
#fig.tight_layout() # improving the space
p = cartesian((np.arange(0,nprows), np.arange(0,npcols)))

for i in npar_labels: #for loop to compute the average by each participant
    df2 = df.loc[df.npar == i,:]
    axes[p[i-1,0],p[i-1,1]].axvline(0, ls='--', color= 'black', alpha=0.2)
    axes[p[i-1,0],p[i-1,1]].axhline(0.5, ls='--', color= 'black', alpha=0.2)
    axes[p[i-1,0],p[i-1,1]].title.set_text(np.unique(df2.npar))
    sns.despine(ax=axes[p[i-1,0],p[i-1,1]],offset=4); # , trim=True
    for cell in nrep_labels:
            #sns.set_palette(mycol)
            sns.regplot(ax=axes[p[i-1,0],p[i-1,1]],x="rDV", y="deci",  data=df2.loc[df2.nrep == cell,:],
               logistic=True, y_jitter=0, scatter_kws={'alpha':0}, ci=True, n_boot=1,  label=cell,  truncate=True, 
                             line_kws ={'alpha':0.95, 'lw':1}); #mean all subject
    

#plt.ylabel('p(diagonal)', fontsize = 25, labelpad=20); plt.yticks(np.arange(0, 1.1, step=0.25), fontsize = 15) 
#axes[1].plt.xlabel('Decision Variable', fontdict={'size':25}, labelpad=20); 
#axes[1].plt.xticks(np.arange(-0.6, 0.61, step=0.3), fontsize = 15)

In [43]:
def log_reg_fit(x, f): # this function was used to calculate the log linear regression between two vectors
    reg = smf.glm(formula = f, data = x, family=sm.families.Binomial()).fit()
    params = reg.params
    PSE = - reg.params.Intercept/reg.params[1]
    # concatenating parameters
    out = pd.DataFrame({'intercept':[params[0]] , 'weight':[params[1]],'PSE': PSE})
    return out #intercept + weight

### Removing bad participants

In [44]:
#df = df.loc[(df.npar != 12) & (df.npar != 15)& (df.npar != 44),:] #she was very distracted and confused the keys#
df.reset_index(drop=True, inplace=True) # reset the row indexes of the pandas dataframe

In [45]:
formula = "deci ~ rDV"
log_par = df.groupby(['npar']).apply(log_reg_fit, formula)
log_par.reset_index(inplace = True)
log_par.head(5)
# here it seems that visual inspection shows that p13 and p15 were pretty bad
log_par[log_par.weight < 0]

#mean_w = np.mean(log_par.weight)
#std_w  = np.std(log_par.weight)
#log_par.weight > (mean_w + 2*std_w)

/var/folders/rx/vfyhhpqd7q172_xk2p75xp7w0000gn/T/ipykernel_20759/4166877942.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  log_par = df.groupby(['npar']).apply(log_reg_fit, formula)
/var/folders/rx/vfyhhpqd7q172_xk2p75xp7w0000gn/T/ipykernel_20759/3843526483.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  PSE = - reg.params.Intercept/reg.params[1]
/var/folders/rx/vfyhhpqd7q172_xk2p75xp7w0000gn/T/ipykernel_20759/3843526483.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior).

,npar,level_1,intercept,weight,PSE
29,30,0,0.020057,-0.192357,0.10427


Masking out subjects with sensitivity smaller than 0.5 or 3 std below the group mean

In [46]:
log_par_avg = np.mean(log_par.weight[0.5 < log_par.weight])
log_par_std = np.std(log_par.weight[0.5 < log_par.weight])

In [47]:
s_outliers  = log_par.npar[log_par.weight < (log_par_avg  - log_par_std*2.5)]
df = df[np.isin(df.npar, s_outliers, invert=True)]

In [51]:
subj_info_df

,npar,gender,age
0,1,F,19
1,2,F,19
2,3,F,19
3,4,F,18
4,5,F,18
5,6,M,17
6,7,F,20
7,8,F,27
8,9,F,22
9,10,F,21


In [48]:
np.shape(subj_info_df)

(38, 3)

In [49]:
np.mean(subj_info_df.age) 

np.float64(20.0)

In [50]:
np.std(subj_info_df.age) 

2.3395906074624073

__Calculating metadprimes__

In [21]:

# Use bayes to compute the metad
def hm_metad(x):
    model, traces = hmetad(
        data=x, 
        stimuli="cond",
        accuracy="correct",
        nRatings=10,
        confidence="confi",
        backend="numpyro")
    out = az.summary(traces, var_names=["meta_d"])
    #az.plot_trace(traces, var_names=["meta_d", "cS1", "cS2"]);
    return out

def mlemetad(x):
    # Assuming 'metad' returns an object with attributes meta_d and m_ratio
    # Replace with your actual metad call if needed
    try:

        # Original line using the actual metad function:
        out_obj = metad(
             data=x, nRatings=10, stimuli='cond', accuracy='correct',
             confidence='confi', verbose=False, padding=True
         )

        # Create the output DataFrame
        out_df = pd.DataFrame(
            [[out_obj['dprime'].item(),out_obj['meta_d'].item(), out_obj['m_ratio'].item()]],
            columns=['dprime','metad', 'm_ratio']
        )
        return out_df
    except Exception as e:
        print(f"Error processing group: {e}")
        # Return DataFrame with NaNs if calculation fails for a group
        return pd.DataFrame([[pd.NA, pd.NA]], columns=['metad', 'm_ratio'])


def sdt(x):
    dprime = x.dprime()
    c = x.criterion()
    out = pd.DataFrame([[dprime, c]], columns=['dprime', 'c'])   
    return out

__Checking which subjects can not be fitted with the model__

Estimating metadprime using MLE

In [503]:
def sdt(data):
    x = pd.DataFrame([])
    x['Stimuli'] = data['cond']
    x['Accuracy'] = data['correct']
    x['Responses'] = data['deci']
    dprime = x.dprime()
    c = x.criterion()
    out = pd.DataFrame([[dprime, c]], columns=['dprime', 'c'])   
    return out

In [504]:
group_sdt = df.groupby(['subj']).apply(sdt)
group_sdt.reset_index(inplace=True)

In [480]:
mle_metadprimes = df.groupby(['subj']).apply(mlemetad).reset_index()



Error processing group: division by zero


Estimating metadprime using a Bayesian approach

In [ ]:
meta_dprimes = df.groupby(['subj']).apply(hm_metad)
meta_dprimes.reset_index(inplace = True)
meta_dprimes['metad_eff'] = meta_dprimes['mean'] / group_sdt['dprime']

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [c1, d1, meta_d, cS1_hn, cS2_hn]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 64 seconds.
There were 37 divergences after tuning. Increase `target_accept` or reparameterize.
There were 70 divergences after tuning. Increase `target_accept` or reparameterize.
There were 107 divergences after tuning. Increase `target_accept` or reparameterize.
There were 53 divergences after tuning. Increase `target_accept` or reparameterize.
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [c1, d1, meta_d, cS1_hn, cS2_hn]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 51 seconds.
There were 23 divergences after tuning. Increase `target_accept` or reparameterize.
There were 52 divergences after tuning. Increase `target_accept` or reparameterize.
There were 37 divergences after tuning. Increase `target_accept` or reparameterize.
There were 45 divergences after tuning. Increase `target_accept` or reparameterize.
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [c1, d1, meta_d, cS1_hn, cS2_hn]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 61 seconds.
There were 42 divergences after tuning. Increase `target_accept` or reparameterize.
There were 56 divergences after tuning. Increase `target_accept` or reparameterize.
There were 54 divergences after tuning. Increase `target_accept` or reparameterize.
There were 76 divergences after tuning. Increase `target_accept` or reparameterize.
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [c1, d1, meta_d, cS1_hn, cS2_hn]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 74 seconds.
There were 15 divergences after tuning. Increase `target_accept` or reparameterize.
There were 28 divergences after tuning. Increase `target_accept` or reparameterize.
There were 14 divergences after tuning. Increase `target_accept` or reparameterize.
There were 7 divergences after tuning. Increase `target_accept` or reparameterize.
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [c1, d1, meta_d, cS1_hn, cS2_hn]


...

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 49 seconds.
There were 23 divergences after tuning. Increase `target_accept` or reparameterize.
There were 42 divergences after tuning. Increase `target_accept` or reparameterize.
There were 48 divergences after tuning. Increase `target_accept` or reparameterize.
There were 80 divergences after tuning. Increase `target_accept` or reparameterize.
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [c1, d1, meta_d, cS1_hn, cS2_hn]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 42 seconds.
There were 8 divergences after tuning. Increase `target_accept` or reparameterize.
There were 2 divergences after tuning. Increase `target_accept` or reparameterize.
There were 17 divergences after tuning. Increase `target_accept` or reparameterize.
There were 20 divergences after tuning. Increase `target_accept` or reparameterize.
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [c1, d1, meta_d, cS1_hn, cS2_hn]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 44 seconds.
There were 30 divergences after tuning. Increase `target_accept` or reparameterize.
There were 8 divergences after tuning. Increase `target_accept` or reparameterize.
There were 16 divergences after tuning. Increase `target_accept` or reparameterize.
There were 12 divergences after tuning. Increase `target_accept` or reparameterize.
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [c1, d1, meta_d, cS1_hn, cS2_hn]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 55 seconds.
There were 92 divergences after tuning. Increase `target_accept` or reparameterize.
There were 52 divergences after tuning. Increase `target_accept` or reparameterize.
There were 131 divergences after tuning. Increase `target_accept` or reparameterize.
There were 99 divergences after tuning. Increase `target_accept` or reparameterize.
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [c1, d1, meta_d, cS1_hn, cS2_hn]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 53 seconds.
There were 179 divergences after tuning. Increase `target_accept` or reparameterize.
There were 203 divergences after tuning. Increase `target_accept` or reparameterize.
There were 311 divergences after tuning. Increase `target_accept` or reparameterize.
The acceptance probability does not match the target. It is 0.692, but should be close to 0.8. Try to increase the number of tuning steps.
There were 241 divergences after tuning. Increase `target_accept` or reparameterize.


In [520]:
# select columns mean and subj
submetaBY = meta_dprimes[['subj', 'mean', 'hdi_3%', 'hdi_97%','metad_eff']]
submetaBY.columns = ['subj', 'metad', 'metad_3%', 'metad_97%','metad_eff']
submetaMLE = mle_metadprimes[['subj', 'metad', 'm_ratio']]
submetaMLE.columns = ['subj', 'metad_MLE', 'm_ratio']
all_metads = pd.DataFrame.merge(submetaBY,submetaMLE,on='subj',how='outer')


df_merged = pd.DataFrame.merge(df,submetaBY,on='subj',how='outer')
df_merged = pd.DataFrame.merge(df_merged ,submetaMLE,on='subj',how='outer')


In [522]:
df_merged.to_csv(os.path.join(results_path, exp_name+'.csv'), index=False)

In [523]:
all_metads.to_csv(os.path.join(results_path, exp_name+'metads.csv'), index=False)

Only run the line below to recalculate metadprimes (takes several hours)

__Metadprime is computed using the library from:__ (only works with pymc <5.0>)

pip install git+https://github.com/LegrandNico/metadPy.git